In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import copy
from datetime import datetime, timedelta
# from keras.utils import to_categorical
# import visualkeras
# import tensorflow as tf
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
# import optuna
# from optuna.samplers import TPESampler
# import keras
# from keras.callbacks import ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
import sys
import os
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import glob 

sys.path.append("/glade/u/home/jhayron/WR_Predictability/3_MLModels/")
# from model_builders_v2 import *
from sklearn import datasets, ensemble
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.utils import class_weight
import json

In [2]:
import xgboost as xgb
from bayes_opt import BayesianOptimization


In [3]:
import os

# Run nvidia-smi to get GPU information
os.system('nvidia-smi')

Wed Jun  5 11:01:43 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 545.23.08              Driver Version: 545.23.08    CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla V100-SXM2-32GB           On  | 00000000:1A:00.0 Off |                    0 |
| N/A   28C    P0              39W / 300W |      3MiB / 32768MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

0

In [4]:
gpu_id = 1

# Load outputs

In [5]:
wr_series = pd.read_csv('/glade/work/jhayron/Data4Predictability/WR_Series_20231120.csv',\
                index_col=0,names=['week0'],skiprows=1,parse_dates=True)
for wk in range(2,10):
    series_temp = copy.deepcopy(wr_series["week0"])
    series_temp.index = series_temp.index - timedelta(weeks = wk-1)
    series_temp.name = f'week{wk-1}'
    if wk==2:
        df_shifts = pd.concat([pd.DataFrame(wr_series["week0"]),pd.DataFrame(series_temp)],axis=1)  
    else:
        df_shifts = pd.concat([df_shifts,pd.DataFrame(series_temp)],axis=1)

# Train model

In [6]:
def get_train_val_test_periods(full_df):
    dic_train_val = {}
    dic_test = {}
    
    start_of_test_periods = np.arange(1981,2021,10)
    end_of_test_periods = start_of_test_periods + 9
    
    for iperiod in range(len(start_of_test_periods)):
        df_test_temp = full_df[str(start_of_test_periods[iperiod]):str(end_of_test_periods[iperiod])]
        df_trainval_temp = full_df.drop(df_test_temp.index)
        
        dic_train_val[start_of_test_periods[iperiod]] = df_trainval_temp
        dic_test[start_of_test_periods[iperiod]] = df_test_temp
    return dic_train_val, dic_test

In [7]:
def xgboost_hyper_param(max_depth,
                        min_child_weight,
                        subsample,
                        colsample_bytree,
                        colsample_bylevel,
                        learning_rate,
                        gamma,
                        reg_lambda,
                        reg_alpha,
                        class_weights):
                        # n_estimators):

    max_depth = int(max_depth)
    # n_estimators = int(n_estimators)
    class_weights = int(np.round(class_weights,0))
    
    clf = xgb.XGBClassifier(n_estimators=100,
                            max_depth=max_depth,
                            learning_rate=learning_rate,
                            subsample=subsample,
                            colsample_bytree=colsample_bytree,
                            colsample_bylevel=colsample_bylevel,
                            gamma=gamma,
                            reg_alpha=reg_alpha,
                            reg_lambda=reg_lambda,
                            num_class=5,
                            tree_method='gpu_hist',
                            gpu_id = gpu_id)
    
    if class_weights == 1:
        cw = class_weight.compute_sample_weight(
            class_weight='balanced',
            y=y_trainval
        )
        scores = cross_val_score(clf, X_trainval, y_trainval, cv=3, scoring='f1_macro'\
                ,fit_params={'sample_weight': cw.tolist()})
    elif class_weights == 0:
        scores = cross_val_score(clf, X_trainval, y_trainval, cv=3, scoring='f1_macro')
        
    return np.mean(scores)


In [8]:
def get_optimized_results(dic_trainval,dic_test):
    pbounds = {
        #### Tree specific hyperparameters ####
        'max_depth': (3,12), #maximum depth of tree
        'min_child_weight': (1,50), #minimum sum of instance weight needed in a child, 
        #prevents the creation of too small leaves
        'subsample': (0.1, 1), ## percentage of samples used for each tree construction
        'colsample_bytree': (0.1, 1), ## percentage of features used for each tree construction.
        'colsample_bylevel': (0.1, 1),## percentage of features used for each split/level.
        #### Learning task-specific hyperparameters ####
        'learning_rate': (0.01, 0.3), #step size shrinkage usage in updates
        'gamma':(0, 3), #minimum loss redution required to make a further partition on a leaf node of tree
        'reg_lambda':(0, 10), #L2 regularization term on weights
        'reg_alpha':(0, 10),#L1 regularization term on weights
        #### General ####
        'class_weights':(0, 1), #use class weights True or False
        # 'n_estimators': (50,51),
        }
    
    start_of_test_periods = np.arange(1981,2021,10)
    list_results = []
    best_params_all = []
    
    for iperiod in range(len(start_of_test_periods)):
        global X_trainval
        global y_trainval
        global X_test
        global y_test
        
        X_trainval = dic_trainval[start_of_test_periods[iperiod]].iloc[:,:-1].values
        y_trainval = dic_trainval[start_of_test_periods[iperiod]].iloc[:,-1]

        X_test = dic_test[start_of_test_periods[iperiod]].iloc[:,:-1].values
        y_test = dic_test[start_of_test_periods[iperiod]].iloc[:,-1]

        optimizer = BayesianOptimization(
            f=xgboost_hyper_param,
            pbounds=pbounds,
            random_state=1,
            verbose=0)
        optimizer.maximize(init_points=50,
                        n_iter=50)
        best_params = optimizer.max['params']

        max_depth = int(best_params['max_depth'])
        # n_estimators = int(best_params['n_estimators'])
        class_weights = int(np.round(best_params['class_weights'],0))

        best_model =xgb.XGBClassifier(n_estimators=300,
                                max_depth=max_depth,
                                learning_rate=best_params['learning_rate'],
                                subsample=best_params['subsample'],
                                colsample_bytree=best_params['colsample_bytree'],
                                colsample_bylevel=best_params['colsample_bylevel'],
                                gamma=best_params['gamma'],
                                reg_alpha=best_params['reg_alpha'],
                                reg_lambda=best_params['reg_lambda'],
                                num_class=5,
                                tree_method='gpu_hist',
                                gpu_id = gpu_id)

        if class_weights == 1:
            cw = class_weight.compute_sample_weight(
                class_weight='balanced',
                y=y_trainval
            )
            best_model.fit(X_trainval,y_trainval,sample_weight=cw)
        elif class_weights == 0:
            best_model.fit(X_trainval,y_trainval)
        y_predicted = best_model.predict(X_test)
        df_results_temp = pd.DataFrame(np.array([y_test.values,y_predicted]).T,
                                       index=y_test.index,
                                       columns=['y_true','y_predicted'])

        list_results.append(df_results_temp)
        print(f1_score(y_test,y_predicted,average='macro'))
        best_params_all.append(best_params)

    return pd.concat(list_results,axis=0), best_params_all


In [9]:
list_folders = np.sort(glob.glob('/glade/u/home/jhayron/WR_Predictability/6_PCA_xgboost/figures_detrended_20240317/*/'))

In [10]:
list_vars = [list_folders[i].split('/')[-2] for i in range(len(list_folders))]

In [11]:
for ivar,var in enumerate(list_vars):
    print(ivar,var)

0 IC_SODA
1 IT_SODA
2 MLD_SODA
3 OHC100_SODA
4 OHC200_SODA
5 OHC300_SODA
6 OHC50_SODA
7 OHC700_SODA
8 OLR_ERA5
9 SD_ERA5
10 SSH_SODA
11 SST_OISSTv2
12 SST_SODA
13 STL_1m_ERA5
14 STL_28cm_ERA5
15 STL_7cm_ERA5
16 STL_full_ERA5
17 SWVL_1m_ERA5
18 SWVL_28cm_ERA5
19 SWVL_7cm_ERA5
20 SWVL_full_ERA5
21 U10_ERA5
22 U200_ERA5
23 Z500_ERA5
24 Z500_ERA5_Region


In [12]:
import warnings

# Ignore XGBoost warnings
warnings.filterwarnings(action='ignore', category=FutureWarning, module='xgboost')

In [ ]:
for ivar in range(6,12):
# for ivar in [11]:
    print(list_vars[ivar])
    combined_df = pd.read_csv(f'{list_folders[ivar]}PC_{list_vars[ivar]}.csv',
                              index_col=0,
                              parse_dates=True)

    time = combined_df.index
    combined_df.index = combined_df.index.floor('D')
    for week_out in range(0,9):
        print(week_out)
        week_out_str = f'week{week_out}'
        fully_combined_df = pd.concat([combined_df,df_shifts[week_out_str]],axis=1)
        fully_combined_df = fully_combined_df.dropna()

        dic_trainval,dic_test = get_train_val_test_periods(fully_combined_df)
        results_temp, best_params_temp = get_optimized_results(dic_trainval,dic_test)
        results_temp.to_csv(f'results/results_{list_vars[ivar]}_{week_out_str}.csv')
        
        # Save the list of dictionaries to a JSON file
        with open(f'best_params/results_{list_vars[ivar]}_{week_out_str}.json', 'w') as json_file:
            json.dump(best_params_temp, json_file)
        

OHC50_SODA
0
0.18428833929140884
0.19636251405844893
0.21584498727075427
0.22961907495083528
1
0.18631873332974017
0.19922158426429343
0.23206997509706623
0.22853663237041758
2
0.1778408209438857
0.2003078679117869
0.19478763702111856
0.19949566027553836
3
0.21327288754831608
0.20763551285245763
0.20071718193409538
0.18864634379065054
4
0.19264895457789782
0.2337249855980708
0.19164417590393296
0.21715105236595594
5
0.20773383911962057
0.20747533434795967
0.2043238159881275
0.22188190655173862
6
0.19059155428879257
0.20509070091840206
0.21454457737879
0.20111413878860357
7
0.17269268304399238
0.19406536243009168
0.2145450485060875
0.1765983635633012
8
0.16086775916452395
